# VisionBridge A-Z base model training

Run this notebook from top to bottom. It downloads the RealSign ISL alphabet dataset from its Git LFS media endpoint, converts images into VisionBridge's 126D landmark representation with the current MediaPipe Tasks API, builds a stratified 80/20 train-validation split from the dataset's training + validation pools, and trains automatically.

The final testing split from the dataset is kept untouched. Training can run for up to 500 epochs and stops early when every A-Z class reaches the configured validation accuracy target.


In [ ]:
%cd /content
!rm -rf /content/VisionBridge
!git clone --depth 1 https://github.com/BharathWaj-K-R/VisionBridge.git /content/VisionBridge
%cd /content/VisionBridge

# Colab already provides the PyTorch training runtime.
# Install only the MediaPipe package used by dataset preprocessing.
%pip -q install --upgrade "mediapipe==0.10.35"

import sys
import urllib.request
from pathlib import Path

import mediapipe as mp
import numpy as np
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)

assert mp.__version__ == "0.10.35"
assert hasattr(mp, "tasks")
assert hasattr(mp.tasks, "vision")

HAND_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
HAND_MODEL_PATH = Path("/content/hand_landmarker.task")

if not HAND_MODEL_PATH.is_file() or HAND_MODEL_PATH.stat().st_size == 0:
    print("Downloading MediaPipe Hand Landmarker model...")
    urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL_PATH)

if HAND_MODEL_PATH.stat().st_size == 0:
    raise RuntimeError("Hand Landmarker model download produced an empty file")

base_options = mp.tasks.BaseOptions(
    model_asset_path=str(HAND_MODEL_PATH)
)
options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(options)
landmarker.close()

print("MediaPipe Tasks Hand Landmarker: PASS")
print("Colab training environment: READY")


In [ ]:
print("Dependency smoke test completed.")
print("Hand model:", HAND_MODEL_PATH)


In [ ]:
from pathlib import Path
import zipfile

DATASET_URL = (
    "https://media.githubusercontent.com/media/RealSign62/"
    "RealSign-Indian-Sign-Language-Dataset/main/Dataset.zip"
)
DATASET_ZIP = Path('/content/RealSign.zip')

!rm -rf /content/RealSign /content/visionbridge_letter_data /content/RealSign.zip
!curl -L --fail --retry 3 -o /content/RealSign.zip "{DATASET_URL}"

if not zipfile.is_zipfile(DATASET_ZIP):
    head = DATASET_ZIP.read_text(errors='replace')[:200]
    raise RuntimeError(
        'RealSign.zip is not a valid ZIP archive. '
        'The downloaded file may be a Git LFS pointer. First bytes: ' + repr(head)
    )

print('RealSign archive size:', DATASET_ZIP.stat().st_size, 'bytes')
!unzip -q /content/RealSign.zip -d /content/RealSign
print('RealSign extraction: PASS')

!python backend/scripts/prepare_letter_dataset.py \
  --input-root /content/RealSign \
  --output-dir /content/visionbridge_letter_data \
  --validation-ratio 0.20 \
  --seed 42 \
  --hand-model-path /content/hand_landmarker.task


In [ ]:
from pathlib import Path

dataset_root = Path('/content/RealSign')
sample_paths = sorted(dataset_root.rglob('*.jpg'))
sample_paths += sorted(dataset_root.rglob('*.jpeg'))
sample_paths += sorted(dataset_root.rglob('*.png'))
if not sample_paths:
    raise RuntimeError('RealSign archive contains no JPG/JPEG/PNG images')

sample = sample_paths[0]
image = mp.Image.create_from_file(str(sample))
detector = mp.tasks.vision.HandLandmarker.create_from_options(options)
result = detector.detect(image)
detector.close()

print('Sample image:', sample)
print('Detected hands in sample:', len(result.hand_landmarks))
print('RealSign + MediaPipe Tasks smoke test: PASS')


In [ ]:
from pathlib import Path
import json
import numpy as np

root = Path('/content/visionbridge_letter_data')
metadata = json.loads((root / 'labels.json').read_text(encoding='utf-8'))
print('Labels:', ''.join(metadata['labels']))
print('Split policy:', metadata['split_policy'])
for split in ('train', 'val', 'test'):
    data = np.load(root / f'{split}.npz')
    print(f'{split}: samples={len(data["x"])} features={data["x"].shape}')


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '/content/VisionBridge/backend')
from app.training.letter_base import train_model

DATA_DIR = Path('/content/visionbridge_letter_data')
OUTPUT = Path('/content/VisionBridge/backend/app/models/weights/letter_base_model.pt')

result = train_model(
    root=DATA_DIR,
    output_path=OUTPUT,
    epochs=500,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-4,
    target_class_accuracy=1.0,
    seed=42,
    hidden_dim=128,
    embedding_dim=64,
    dropout=0.10,
)

print('\nTraining result:')
print('target_reached =', result['reached_target'])
print('test_accuracy =', f"{result['test_accuracy']:.4f}")


In [ ]:
import sys
sys.path.insert(0, '/content/VisionBridge/backend')
from app.models.letter_model import load_checkpoint

checkpoint = '/content/VisionBridge/backend/app/models/weights/letter_base_model.pt'
model = load_checkpoint(checkpoint)
print('CHECKPOINT LOAD: PASS')
print('input_dim =', model.input_dim)
print('hidden_dim =', model.hidden_dim)
print('embedding_dim =', model.embedding_dim)
print('classes =', model.num_classes)
print('labels =', ''.join(model.labels))


## After the run

The trained checkpoint is saved at `backend/app/models/weights/letter_base_model.pt`. Download that file into your local VisionBridge repository at the same path.

The stopping condition is based on the per-letter validation split. The final test accuracy is reported separately so the test set remains an honest held-out measurement.
